### Contract Data

This stage uploads pre-generated vendor supply contract PDFs to a Unity Catalog volume
and loads the structured metadata as Delta tables. Contracts are the authoritative source
for payment terms, price schedules, volume discount tiers, SLA requirements, and penalty clauses.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create schema and volume for contract documents

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.procurement")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.procurement.contracts")
print(f"✅ Created volume {CATALOG}.procurement.contracts")

##### Copy contract PDFs to the Unity Catalog volume

In [ ]:
import os
import glob

pdf_source_dir = os.path.abspath("../data/contracts/pdfs")
volume_path = f"/Volumes/{CATALOG}/procurement/contracts"

pdf_files = glob.glob(os.path.join(pdf_source_dir, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files to upload")

for pdf_file in pdf_files:
    filename = os.path.basename(pdf_file)
    with open(pdf_file, "rb") as src:
        with open(f"{volume_path}/{filename}", "wb") as dst:
            dst.write(src.read())
    print(f"  Uploaded: {filename}")

print(f"✅ Uploaded {len(pdf_files)} contract PDFs to {volume_path}")

##### Load contract metadata into Delta tables

In [ ]:
import json
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)

metadata_path = os.path.abspath("../data/contracts/contract_metadata.json")
with open(metadata_path) as f:
    metadata = json.load(f)

print(f"Loaded {len(metadata['contracts'])} contracts")

In [ ]:
# --- Contracts table (one row per contract) ---
contract_rows = []
for c in metadata["contracts"]:
    contract_rows.append({
        "contract_id": c["contract_id"],
        "supplier_id": c["supplier_id"],
        "supplier_name": c["supplier_name"],
        "category": c["category"],
        "effective_date": c["effective_date"],
        "expiration_date": c["expiration_date"],
        "payment_terms": c["payment_terms"],
        "payment_terms_detail": c["payment_terms_detail"],
        "late_payment_penalty": c["late_payment_penalty"],
        "price_stability_clause": c["price_stability_clause"],
        "dispute_resolution": c["dispute_resolution"],
        "has_sla": c.get("sla") is not None,
        "sla_delivery_days": int(c["sla"]["delivery_days"]) if c.get("sla") else None,
        "sla_penalty_description": c["sla"]["penalty"] if c.get("sla") else None,
        "has_volume_discount": c.get("volume_discounts") is not None,
        "pdf_filename": c["pdf_filename"],
    })

contract_schema = StructType([
    StructField("contract_id", StringType()),
    StructField("supplier_id", StringType()),
    StructField("supplier_name", StringType()),
    StructField("category", StringType()),
    StructField("effective_date", StringType()),
    StructField("expiration_date", StringType()),
    StructField("payment_terms", StringType()),
    StructField("payment_terms_detail", StringType()),
    StructField("late_payment_penalty", StringType()),
    StructField("price_stability_clause", StringType()),
    StructField("dispute_resolution", StringType()),
    StructField("has_sla", StringType()),
    StructField("sla_delivery_days", StringType()),
    StructField("sla_penalty_description", StringType()),
    StructField("has_volume_discount", StringType()),
    StructField("pdf_filename", StringType()),
])

df_contracts = spark.createDataFrame(contract_rows)
df_contracts.write.mode("overwrite").saveAsTable(f"{CATALOG}.procurement.contracts_metadata")
print(f"✅ Created contracts_metadata table with {df_contracts.count()} rows")

In [ ]:
# --- Contract price schedule (one row per product per contract) ---
price_rows = []
for c in metadata["contracts"]:
    for item in c.get("price_schedule", []):
        price_rows.append({
            "contract_id": c["contract_id"],
            "supplier_id": c["supplier_id"],
            "supplier_name": c["supplier_name"],
            "category": c["category"],
            "item": item["item"],
            "unit": item["unit"],
            "contract_unit_price": float(item["price"]),
        })

price_schema = StructType([
    StructField("contract_id", StringType()),
    StructField("supplier_id", StringType()),
    StructField("supplier_name", StringType()),
    StructField("category", StringType()),
    StructField("item", StringType()),
    StructField("unit", StringType()),
    StructField("contract_unit_price", DoubleType()),
])

df_prices = spark.createDataFrame(price_rows, schema=price_schema)
df_prices.write.mode("overwrite").saveAsTable(f"{CATALOG}.procurement.contract_price_schedule")
print(f"✅ Created contract_price_schedule table with {df_prices.count()} rows")

##### Register resources with uc_state for cleanup

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

print("✅ Contract data stage complete")